# Step 4: SQL-Based Business Analysis
Ingests cleaned data into SQL database and performs queries using Joins, Aggregations, CTEs, Window Functions, and Subqueries.


In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = os.getenv("MYSQL_PORT", "3306")
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "root")
MYSQL_DB = os.getenv("MYSQL_DB", "cart2insights_db")

mysql_uri = (
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}"
)
engine = create_engine(mysql_uri)
print(f"Connected to MySQL database: {MYSQL_DB} on {MYSQL_HOST}:{MYSQL_PORT}")

query_cte = """
WITH CustomerSpend AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count,
        SUM(oi.price + oi.freight_value) AS total_spent
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_unique_id
)
SELECT 
    customer_unique_id,
    order_count,
    ROUND(total_spent, 2) AS total_spent,
    RANK() OVER (ORDER BY total_spent DESC) AS spending_rank
FROM CustomerSpend
LIMIT 10;
"""

df = pd.read_sql(text(query_cte), engine)
print(df, flush=True)


Connected to MySQL database: cart2insights_db on localhost:3306
                 customer_unique_id  order_count  total_spent  spending_rank
0  0a0a92112bd4c708ca5fde585afaa872            1     13664.08              1
1  da122df9eeddfedc1dc1f5349a1a690c            2      7571.63              2
2  763c8b1c9c68a0229c42c9fc6f662b93            1      7274.88              3
3  dc4802a71eae9be1dd28f5d788ceb526            1      6929.31              4
4  459bef486812aa25204be022145caa62            1      6922.21              5
5  ff4159b92c40ebe40454e3e6a7c35ed6            1      6726.66              6
6  4007669dec559734d6f53e029e360987            1      6081.54              7
7  5d0a2980b292d049061542014e8960bf            1      4809.44              8
8  eebb5dda148d3893cdaf5b5ca3040ccb            1      4764.34              9
9  48e1ac109decbb87765a3eade6854098            1      4681.78             10
